In [1]:
import sqlite3
from datetime import datetime, timezone, timedelta

In [2]:
# Login and signup page
def login_signup():
    if check_session():
        return

    your_username = input("Insert username: ")

    cursor1.execute("""
        SELECT EXISTS(SELECT * FROM profiles WHERE username = ?)
        """, (your_username,))
    username_exists = cursor1.fetchone()[0]

    if username_exists:
        cursor1.execute("""
            SELECT * FROM profiles WHERE username = ?
            """, (your_username,))
        query = cursor1.fetchone()
        your_name = query[2]
        your_timezone = query[3]
        print(f"Welcome back {your_name}!")
    else:
        print("Redirecting to sign up page...")
        your_name = input("Insert name: ")
        your_timezone = input("Your timezone is: GMT+")
        
        query = [(your_username, your_name, your_timezone)]
        cursor1.executemany("""
            INSERT INTO profiles (username, name, timezone)
            VALUES (?, ?, ?)
            """, query)
        connection1.commit()
        print(f"Sign up complete! Hello {your_name}!")
    global session_info
    session_info = [your_username, your_name, your_timezone, int(datetime.now().timestamp())]
    checkschedules()

# Check login info:
def check_session():
    if session_info == []:
        print("You have been logged out!")
    elif (int(datetime.now().timestamp()) - session_info[3]) > 36000:
        wipe_session()
        print("Session expired! You have been logged out.")
    else:
        return True

# Clear login info:
def wipe_session():
    session_info = []

In [3]:
# Get session timezone info
def session_tz():
    return timezone(timedelta(hours = session_info[2]))

def schedule_err(hour):
    wib_rel = 7 - session_info[2] 
    if 8 <= hour < 17 and 8 <= (hour + wib_rel) < 17:
        return False
    else:
        return True

# Check schedule function
def checkschedules():
    cursor2.execute("""
        SELECT EXISTS(SELECT * FROM schedules WHERE username = ?)
        """, (session_info[0],))
    schedules_exists = cursor2.fetchone()[0]

    if schedules_exists:
        cursor2.execute("""
        SELECT * FROM schedules WHERE username = ?
        """, (session_info[0],))
        allschedules = cursor2.fetchall()
        print(f"{session_info[1]}, you have an appointment in:")
        for appointments in allschedules:
            dt = datetime.fromtimestamp(appointments[2], tz = session_tz())
            print(dt)
        print("according to your local time.")
    else:
        print(f"{session_info[1]}, you don't have any appointments scheduled.")

# Query new schedule function
def schedule():
    print("Redirecting to scheduling page...")
    in_year = int(input("Year: "))
    in_month = int(input("Month: "))
    in_day = int(input("Day: "))
    in_hour = int(input("Hour (24-hour format): "))
    in_timestamp = int(datetime(in_year, in_month, in_day, in_hour, 0, 0, tzinfo = session_tz()).timestamp())

    if in_timestamp > int(datetime.now().timestamp()):
        if schedule_err(in_hour) == False:
            query = [(session_info[0], in_timestamp)]
            cursor2.executemany("""
                INSERT INTO schedules (username, timestamp)
                VALUES (?, ?)
                """, query)
            connection2.commit()
            print("New appointment added!")
        else:
            print("Scheduled time out of established working hours! Please try again.")
    else:
        print("Scheduling error! Please try again.")

In [4]:
# Initalize connection with profiles
profiles_db = "profiles.db"
connection1 = sqlite3.connect(profiles_db)
cursor1 = connection1.cursor()
cursor1.execute("""
    CREATE TABLE IF NOT EXISTS profiles (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT NOT NULL,
        name TEXT NOT NULL,
        timezone INTEGER NOT NULL)
    """)

# Initialize connection with schedules
schedules_db = "schedules.db"
connection2 = sqlite3.connect(schedules_db)
cursor2 = connection2.cursor()
cursor2.execute("""
    CREATE TABLE IF NOT EXISTS schedules (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT NOT NULL,
        timestamp INTEGER NOT NULL)
    """)

# Initalize session
session_info = []

In [5]:
# Main page
login_signup()

You have been logged out!
Welcome back Liliana Aoyama!
Liliana Aoyama, you have an appointment in:
2026-10-12 16:00:00+08:00
2026-10-23 09:00:00+08:00
according to your local time.


In [6]:
# Query a schedule
schedule()

Redirecting to scheduling page...
New appointment added!


In [ ]:
# Stop connections
cursor1.close()
connection1.close()
cursor2.close()
connection2.close()

# Delete the database
import os

databasename = schedules_db
if os.path.exists(databasename):
    os.remove(databasename)
    print("Database removed!")
else:
    print("Database not found :(")